# G1 拟合目标短测

在原来每个种子的八份拟合文档内按清单顺序取前四份做拟合，后四份测迁移。两种求解器、两种目标共用原随机坐标表的前 32 对。默认只显示选项。详细定义和判据见 `docs/G1_OBJECTIVE_TRIAL.md`。

In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

RUN_SCREEN = False
CONFIRM_IF_PROMISING = False
RUN_SCREEN = RUN_SCREEN or os.environ.pop("CELLQUANT_SCREEN_OBJECTIVE", "") == "1"
CONFIRM_IF_PROMISING = CONFIRM_IF_PROMISING or os.environ.pop("CELLQUANT_CONFIRM_OBJECTIVE", "") == "1"
if RUN_SCREEN:
    hint = os.environ.get("CELLQUANT_PROJECT_ROOT")
    roots = [Path(hint)] if hint else [Path.cwd(), *Path.cwd().parents, Path("/content/CellQuant")]
    root = next((p.resolve() for p in roots if (p / "src/cellquant/objective_screen.py").is_file()), None)
    if root is None:
        raise FileNotFoundError("请将本轮更新解压至 01 的 checkout。")
    cfg = root / os.environ.get("CELLQUANT_PILOT_CONFIG", "runs/prepared/g1_gpu/config.json")
    if not cfg.is_file():
        raise FileNotFoundError(f"请恢复或准备原始 G1 输入清单 {cfg}")
    source = str(root / "src")
    sys.path.insert(0, source)
    os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [source, os.environ.get("PYTHONPATH")]))
    work = root / "runs/01_pilot" / ("objective_" + uuid4().hex[:12])
    work.mkdir(parents=True)
else:
    print("短测待运行。在准备好的 A100 上开启 RUN_SCREEN。")

In [ ]:
if RUN_SCREEN:
    from zipfile import ZipFile, ZIP_DEFLATED
    from cellquant.runtime import read_json, stream_command

    def launch(label, config, flags):
        index = work / (label + "_index.json")
        command = [sys.executable, "-u", "-m", "cellquant.experiment", "--config", str(config),
                   *flags, "--result-index", str(index)]
        print("本次日志", work / (label + ".log"), flush=True)
        job = stream_command(command, work / (label + ".log"), cwd=root, timeout_s=5*3600)
        if not index.is_file():
            raise RuntimeError(f"未生成运行索引，详见 {job['log_path']}")
        run = Path(read_json(index)["run_dir"])
        result = read_json(run / "result.json")
        archive = work / (label + "_" + run.name + ".zip")
        patterns = ("result.json", "config.json", "execution_fingerprint.json", "timings.json", "resources.json",
                    "objective_screen_summary.json", "objective_screen/seed_*/*/*/objective_result.json",
                    "G1*.json", "including_original_grid/G1*.json", "seed_*/*/pilot_result.json",
                    "seed_*/*/original_grid_result.json")
        with ZipFile(archive, "w", ZIP_DEFLATED) as z:
            for pattern in patterns:
                for path in sorted(run.glob(pattern)):
                    z.write(path, path.relative_to(run))
        print("结果包", archive)
        if job["returncode"]:
            raise RuntimeError(result.get("reason") or result.get("readiness", {}).get("missing") or result["status"])
        return run, result

    screen_run, screen = launch("screen", cfg, ["--screen-objective"])
    decision = screen["objective_screen"]["decision"]
    print("短测判定", decision)
    for name, v in screen["objective_screen"].get("comparisons", {}).items():
        print(name, round(v["median_site_gain_pct"], 4), "%", v["seed_median_gain_pct"])

In [ ]:
if RUN_SCREEN and CONFIRM_IF_PROMISING and decision == "TEST_ONE_G1":
    destination = work / "confirmation_inputs"
    prepare = [sys.executable, "-u", "scripts/prepare_experiment.py", "--config", str(cfg),
               "--relative-g1-screen", str(screen_run), "--output", str(destination)]
    job = stream_command(prepare, work / "prepare_confirmation.log", cwd=root, timeout_s=3600)
    if job["returncode"]:
        raise RuntimeError(f"确认文档准备失败，详见 {job['log_path']}")
    confirm_run, confirmation = launch("confirmation", destination / "config.json", [])
    print("正式 G1", read_json(confirm_run / "G1.json")["status"],
          "加入原目标网格后的 G1", read_json(confirm_run / "including_original_grid/G1.json")["status"])
    print("G2 使用的证据", confirm_run / "including_original_grid/G1_evidence.json")